In [166]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score,f1_score,classification_report
from xgboost import XGBClassifier

from sklearn.preprocessing import LabelEncoder

In [167]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")  #loading the dataset

In [168]:
df.shape

(7043, 21)

In [169]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [170]:
print(df.dtypes)

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


In [171]:

df.drop("customerID", axis=1, inplace=True)  #no relation of customer_id with churn,hence removing it

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce") #Converting TotalCharges to numeric



df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)
#if there are blank or NaN values, replacing then with median values

C:\Users\mrunm\AppData\Local\Temp\ipykernel_4480\3921605871.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: TotalCharges, Length: 7043, dtype: float64

In [172]:
#Converting text data into numerical data using label encoder


le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == "object" or df[col].dtype == "string":
        df[col] = le.fit_transform(df[col])



In [173]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


### Splitting Features And Targets

In [174]:
X = df.drop("Churn", axis=1)  # x contains input features
# axis = 1removes churn column from x

y = df["Churn"]  #Y is target/output

In [175]:
np.unique(y, return_counts = True)

(array([0, 1]), array([5174, 1869]))

### Train-Test Split

In [176]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
    
)

In [177]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5634, 19), (1409, 19), (5634,), (1409,))

In [212]:
df.head(3)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1


In [215]:
df.shape

(7043, 20)

### Feature Scaling

In [178]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Hyperparameter Tuning

In [179]:
params_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.01, 0.1, 0.3], # Speed of model
    "subsample": [0.8, 1.0],  # how much percentage of data to be used whil training
    "colsample_bytree": [0.8, 1.0]  #decides how much features to be used 
}

### Grid Search

In [180]:
xgb_tuning = GridSearchCV(
    XGBClassifier(
        random_state=42,
        eval_metric="logloss"  #Training ke time model ki error measure karta hai  , more logloss = more errors
    ),
    param_grid=params_xgb,
    cv=5,  # Divides data into 5 parts
    scoring="f1",  #uses accuracy to select the best model
    n_jobs=-1  #Computer ke saare CPU cores use karo.(for faster training)
)

xgb_tuning.fit(X_train_scaled, y_train)

# Training Accuracy
train_pred = xgb_tuning.predict(X_train_scaled)

print("Training Accuracy")
print(accuracy_score(y_train, train_pred))

Training Accuracy
0.8116790912318069


In [203]:
xgb_tuning.best_params_

{'colsample_bytree': 0.8,
 'learning_rate': 0.1,
 'max_depth': 2,
 'n_estimators': 100,
 'subsample': 0.8}

In [196]:
xgb_tuning.score(X_train_scaled, y_train)

0.6042521447221186

In [204]:
y_pred = xgb_tuning.predict(X_test_scaled)

In [206]:
f1_score(y_test, y_pred)

0.6062407132243685

In [209]:
y_pred[20:31]

array([1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0])

In [210]:
y_test[20:31].to_numpy()

array([0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])

#### Best Parameters

In [181]:
print("Best Parameters:")
print(xgb_tuning.best_params_)

Best Parameters:
{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 100, 'subsample': 0.8}


### Predictions

In [182]:
y_pred = xgb_tuning.predict(X_test_scaled)

print("Testing Accuracy")
print(accuracy_score(y_test, y_pred))

Testing Accuracy
0.8119233498935415


### Precision

In [183]:
precision = precision_score(
    y_test,
    y_pred,
    average="weighted"
)

print("Precision:", precision)

Precision: 0.8032394864650216


### Classification Report

In [184]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.91      0.88      1036
           1       0.68      0.55      0.61       373

    accuracy                           0.81      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [185]:
sample = X_test_scaled[0:1]

prediction = xgb_tuning.predict(sample)

if prediction[0] == 0:
    print("Customer Will Stay")
else:
    print("Customer Will Leave")

Customer Will Leave


### Saving Model

In [186]:
import joblib

joblib.dump(
    xgb_tuning.best_estimator_,
    "xgboost_model.pkl"
)

['xgboost_model.pkl']

In [ ]:
pt_model.score(X_train, y_train)         #train accuracy

0.7552360667376642

In [191]:
np.unique(y_hat, return_counts = True)

(array([0, 1]), array([1348,   61]))

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_hat)            #test accuracy

0.765791341376863

In [194]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_hat))

              precision    recall  f1-score   support

           0       0.76      0.99      0.86      1036
           1       0.85      0.14      0.24       373

    accuracy                           0.77      1409
   macro avg       0.81      0.57      0.55      1409
weighted avg       0.79      0.77      0.70      1409

